In [1]:
!pip install -q -U langchain langchain-community langchain-ollama

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
!apt-get update -qq
!apt-get install -y -qq zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 122797 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [3]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [27]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)

print("Ollama server started")

Ollama server started


In [5]:
!ollama pull llama3.2:3b

In [6]:
!ollama list

NAME           ID              SIZE      MODIFIED      
llama3.2:3b    a80c4f17acd5    2.0 GB    9 seconds ago    


In [7]:
from langchain_core.prompts import PromptTemplate
from langchain_ollama import OllamaLLM
from langchain_core.output_parsers import StrOutputParser

In [8]:
prompt = PromptTemplate(
    input_variables=["question"],
    template="""
You are a helpful AI assistant.

Answer the following question clearly and briefly:

Question: {question}

Answer:
"""
)

llm = OllamaLLM(model="llama3.2:3b")

parser = StrOutputParser()

chain = prompt | llm | parser

print("LangChain chain created successfully")

LangChain chain created successfully


In [9]:
questions = [
    "What is artificial intelligence?",
    "What is machine learning?",
    "What is deep learning?",
    "What is natural language processing?",
    "What is generative AI?"
]

for i, question in enumerate(questions, 1):
    response = chain.invoke({"question": question})

    print(f"\n--- Test {i} ---")
    print("Question:", question)
    print("Answer:", response)


--- Test 1 ---
Question: What is artificial intelligence?
Answer: Artificial intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think like humans and mimic their actions. It involves developing algorithms and statistical models to enable machines to perform tasks that typically require human intelligence, such as learning, problem-solving, and decision-making.

--- Test 2 ---
Question: What is machine learning?
Answer: Machine learning is a type of artificial intelligence that enables computers to learn from data, identify patterns, and make predictions or decisions without being explicitly programmed. It involves algorithms that automatically improve their performance on a task by learning from experience, rather than being manually programmed.

--- Test 3 ---
Question: What is deep learning?
Answer: Deep learning is a subset of machine learning that uses artificial neural networks with multiple layers to analyze and interpret data. Th

In [11]:
conversation_history = []

print("Conversation memory created")

Conversation memory created


In [12]:
memory_prompt = PromptTemplate(
    input_variables=["history", "question"],
    template="""
You are a helpful conversational assistant.

Conversation history:
{history}

Current question:
{question}

Answer the current question while considering the conversation history.
"""
)

memory_chain = memory_prompt | llm | parser

print("Memory-enabled chain created")

Memory-enabled chain created


In [13]:
conversation = [
    "My name is Bhavana.",
    "I am studying Information Science.",
    "What branch am I studying?",
    "What is a good AI skill for my field?",
    "Can you summarize what you know about my study background from this conversation?"
]

for i, question in enumerate(conversation, 1):

    history_text = "\n".join(conversation_history)

    response = memory_chain.invoke({
        "history": history_text,
        "question": question
    })

    conversation_history.append(f"User: {question}")
    conversation_history.append(f"Assistant: {response}")

    print(f"\n--- Turn {i} ---")
    print("User:", question)
    print("Assistant:", response)


--- Turn 1 ---
User: My name is Bhavana.
Assistant: Hello Bhavana! It's nice to meet you. Since we just started our conversation, I don't have any specific information to build upon. Would you like to share something about yourself, or is there something on your mind that you'd like to talk about? I'm all ears!

--- Turn 2 ---
User: I am studying Information Science.
Assistant: Hello Bhavana! It's great to continue our conversation. I noticed you mentioned that you're studying Information Science. That's a fascinating field! Information Science is a multidisciplinary field that combines aspects of computer science, information technology, and library science.

As someone who's studying Information Science, I'm sure you're exploring various topics such as data management, information retrieval, and human-computer interaction. You might be interested in learning about the latest trends and applications in the field, such as artificial intelligence, natural language processing, or data v

In [14]:
print("Conversation History")
print("=" * 50)

for message in conversation_history:
    print(message)

Conversation History
User: My name is Bhavana.
Assistant: Hello Bhavana! It's nice to meet you. Since we just started our conversation, I don't have any specific information to build upon. Would you like to share something about yourself, or is there something on your mind that you'd like to talk about? I'm all ears!
User: I am studying Information Science.
Assistant: Hello Bhavana! It's great to continue our conversation. I noticed you mentioned that you're studying Information Science. That's a fascinating field! Information Science is a multidisciplinary field that combines aspects of computer science, information technology, and library science.

As someone who's studying Information Science, I'm sure you're exploring various topics such as data management, information retrieval, and human-computer interaction. You might be interested in learning about the latest trends and applications in the field, such as artificial intelligence, natural language processing, or data visualizatio

In [15]:
from langchain_core.tools import tool

@tool
def web_search(query: str) -> str:
    """A simple web search stub that returns simulated search information."""

    return f"Web search result for '{query}': This is a simulated search result for demonstration."

@tool
def calculator(expression: str) -> str:
    """Calculate a mathematical expression."""

    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception:
        return "Invalid mathematical expression."

tools = [web_search, calculator]

print("Tools created:")
for tool in tools:
    print("-", tool.name)

Tools created:
- web_search
- calculator


In [16]:
print(calculator.invoke("25 * 4"))

100


In [17]:
print(web_search.invoke("latest AI trends"))

Web search result for 'latest AI trends': This is a simulated search result for demonstration.


In [19]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful AI assistant. Use the available tools when necessary."
)

print("Two-tool agent created successfully")

Two-tool agent created successfully


In [22]:
from langchain_ollama import ChatOllama

chat_llm = ChatOllama(
    model="llama3.2:3b"
)

print("ChatOllama created successfully")

ChatOllama created successfully


In [23]:
from langchain.agents import create_agent

agent = create_agent(
    model=chat_llm,
    tools=tools,
    system_prompt="You are a helpful AI assistant. Use the available tools when necessary."
)

print("Two-tool agent created successfully")

Two-tool agent created successfully


In [28]:
tasks = [
    "Calculate 125 * 8",
    "Calculate 450 / 9 + 20",
    "Search for information about artificial intelligence"
]

for i, task in enumerate(tasks, 1):

    print(f"\n{'=' * 50}")
    print(f"Task {i}: {task}")
    print("=" * 50)

    result = agent.invoke({
        "messages": [
            {"role": "user", "content": task}
        ]
    })

    final_message = result["messages"][-1]

    print("\nFinal Answer:")
    print(final_message.content)


Task 1: Calculate 125 * 8

Final Answer:
The result of 125 * 8 is 1000.

Task 2: Calculate 450 / 9 + 20

Final Answer:
The result of the calculation 450 / 9 + 20 is 70.0.

Task 3: Search for information about artificial intelligence

Final Answer:
Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as learning, problem-solving, and decision-making.

There are several types of AI, including:

1. **Narrow or Weak AI**: This type of AI is designed to perform a specific task, such as facial recognition or language translation.
2. **General or Strong AI**: This type of AI is designed to perform any intellectual task that a human can, and is still a subject of ongoing research and development.
3. **Superintelligence**: This type of AI is significantly more intelligent than the best human minds, and is still a topic of debate and speculation.

Some of the key benefits of AI include:

* **Increased e